In [1]:
import logging
import pandas as pd
from p2p.client import P2PClient # HTTP wrapper: talks to the Rust node
from p2p.dataset import P2PDataset # Cache + query layer: files(), load(), query(), federate()

# Show INFO-level logs (cache hits/misses, fetch calls) directly in the notebook
logging.basicConfig(level=logging.INFO)

# Connect to the local Rust node
client = P2PClient("http://localhost:8080")
# High-level entry point: everything below goes through this object
dataset = P2PDataset(client)

# See wha'ts available on the network
print(dataset.files())

[{'file_name': 'yellow_tripdata_2026-01.parquet', 'institution': 'mtl', 'num_rows': 3724889, 'num_row_groups': 4, 'file_size_bytes': 64165080, 'columns': [{'c_type': 'INT32', 'name': 'VendorID'}, {'c_type': 'INT64', 'name': 'tpep_pickup_datetime'}, {'c_type': 'INT64', 'name': 'tpep_dropoff_datetime'}, {'c_type': 'INT64', 'name': 'passenger_count'}, {'c_type': 'DOUBLE', 'name': 'trip_distance'}, {'c_type': 'INT64', 'name': 'RatecodeID'}, {'c_type': 'BYTE_ARRAY', 'name': 'store_and_fwd_flag'}, {'c_type': 'INT32', 'name': 'PULocationID'}, {'c_type': 'INT32', 'name': 'DOLocationID'}, {'c_type': 'INT64', 'name': 'payment_type'}, {'c_type': 'DOUBLE', 'name': 'fare_amount'}, {'c_type': 'DOUBLE', 'name': 'extra'}, {'c_type': 'DOUBLE', 'name': 'mta_tax'}, {'c_type': 'DOUBLE', 'name': 'tip_amount'}, {'c_type': 'DOUBLE', 'name': 'tolls_amount'}, {'c_type': 'DOUBLE', 'name': 'improvement_surcharge'}, {'c_type': 'DOUBLE', 'name': 'total_amount'}, {'c_type': 'DOUBLE', 'name': 'congestion_surcharge'}

In [2]:
dataset.files_df()

,file_name,institution,num_rows,num_row_groups,size,columns
0,yellow_tripdata_2026-01.parquet,mtl,3724889,4,61.2 MB,"VendorID, tpep_pickup_datetime, tpep_dropoff_d..."
1,yellow_tripdata_2026-01.parquet,peer1,3724889,4,61.2 MB,"VendorID, tpep_pickup_datetime, tpep_dropoff_d..."
2,yellow_tripdata_2026-02.parquet,suisse,3399866,4,56.0 MB,"VendorID, tpep_pickup_datetime, tpep_dropoff_d..."


In [4]:
dataset.files_df(2)

,file_name,institution,num_rows,num_row_groups,size,columns
0,yellow_tripdata_2026-02.parquet,peer1,3399866,4,56.0 MB,"VendorID, tpep_pickup_datetime, ... (+18 more)"
1,yellow_tripdata_2026-01.parquet,peer1,3724889,4,61.2 MB,"VendorID, tpep_pickup_datetime, ... (+18 more)"


In [4]:
# load(): single file as a DataFrame
df = dataset.load("yellow_tripdata_2026-02.parquet")
df.head()

INFO:p2p.dataset:fetching 'yellow_tripdata_2026-02.parquet' from suisse ...
INFO:p2p.dataset:saved -> /home/vince/bthesis/vincent-cordola-bthesis-p2p_dataset_federation/peer-dataset-federation/node/cache/106c59bb36ce157d.parquet
INFO:p2p.dataset:loading 'yellow_tripdata_2026-02.parquet' into dataframe


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,7,2026-02-01 00:05:57,2026-02-01 00:05:57,1.0,0.94,1.0,N,107,170,1,7.2,0.00,0.5,0.00,0.0,1.0,12.95,2.5,0.00,0.75
1,7,2026-02-01 00:35:58,2026-02-01 00:35:58,1.0,1.93,1.0,N,234,141,1,11.4,0.00,0.5,3.43,0.0,1.0,20.58,2.5,0.00,0.75
2,2,2026-02-01 00:08:41,2026-02-01 00:39:32,1.0,9.99,1.0,N,138,68,1,44.3,6.00,0.5,11.01,0.0,1.0,67.81,2.5,1.75,0.75
3,1,2026-02-01 00:29:06,2026-02-01 00:41:04,0.0,1.70,1.0,N,209,13,1,12.8,4.25,0.5,3.70,0.0,1.0,22.25,2.5,0.00,0.75
4,1,2026-02-01 00:53:52,2026-02-01 01:11:21,0.0,3.70,1.0,N,249,229,1,19.8,4.25,0.5,6.35,0.0,1.0,31.90,2.5,0.00,0.75


In [35]:
results = dataset.query("iris.parquet","titanic.parquet","test.parquet")

INFO:p2p.dataset:cache hit: iris.parquet
INFO:p2p.dataset:cache hit: titanic.parquet


In [36]:
for name, df in results.items():
    print(name, df.shape)

iris.parquet (150, 5)
titanic.parquet (891, 12)


In [37]:
con = dataset.federate("yellow_tripdata_2026-01.parquet","yellow_tripdata_2026-02.parquet")

df2 = con.sql("""SELECT * FROM dataset WHERE "passenger_count" > 2 """).df()

df2

INFO:p2p.dataset:cache hit: yellow_tripdata_2026-01.parquet
INFO:p2p.dataset:cache hit: yellow_tripdata_2026-02.parquet
INFO:p2p.dataset:federated view 'dataset' created from 2 file(s)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,38.7,1.0,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
1,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3,1.83,1,N,237,263,1,10.7,1.0,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00
2,2,2026-01-01 00:42:31,2026-01-01 00:44:28,3,0.33,1,N,234,90,2,4.4,1.0,0.5,0.00,0.0,1.0,10.15,2.5,0.0,0.75
3,2,2026-01-01 00:10:51,2026-01-01 00:18:46,4,1.05,1,N,164,164,1,9.3,1.0,0.5,3.01,0.0,1.0,18.06,2.5,0.0,0.75
4,2,2026-01-01 00:28:29,2026-01-01 00:46:04,3,4.78,1,N,238,243,1,23.3,1.0,0.5,2.00,0.0,1.0,27.80,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242507,2,2026-02-28 23:06:44,2026-02-28 23:31:36,3,3.70,1,N,79,140,1,24.0,1.0,0.5,5.95,0.0,1.0,35.70,2.5,0.0,0.75
242508,2,2026-02-28 23:15:21,2026-02-28 23:32:53,3,1.55,1,N,48,170,1,15.6,1.0,0.5,3.20,0.0,1.0,24.55,2.5,0.0,0.75
242509,2,2026-02-28 23:52:49,2026-03-01 00:03:09,3,1.63,1,N,186,107,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75
242510,2,2026-02-28 23:34:43,2026-02-28 23:50:47,3,1.02,1,N,79,249,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75


In [38]:
df2.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,38.7,1.0,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
1,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3,1.83,1,N,237,263,1,10.7,1.0,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00
2,2,2026-01-01 00:42:31,2026-01-01 00:44:28,3,0.33,1,N,234,90,2,4.4,1.0,0.5,0.00,0.0,1.0,10.15,2.5,0.0,0.75
3,2,2026-01-01 00:10:51,2026-01-01 00:18:46,4,1.05,1,N,164,164,1,9.3,1.0,0.5,3.01,0.0,1.0,18.06,2.5,0.0,0.75
4,2,2026-01-01 00:28:29,2026-01-01 00:46:04,3,4.78,1,N,238,243,1,23.3,1.0,0.5,2.00,0.0,1.0,27.80,0.0,0.0,0.00


In [39]:
df2.tail()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
242507,2,2026-02-28 23:06:44,2026-02-28 23:31:36,3,3.70,1,N,79,140,1,24.0,1.0,0.5,5.95,0.0,1.0,35.70,2.5,0.0,0.75
242508,2,2026-02-28 23:15:21,2026-02-28 23:32:53,3,1.55,1,N,48,170,1,15.6,1.0,0.5,3.20,0.0,1.0,24.55,2.5,0.0,0.75
242509,2,2026-02-28 23:52:49,2026-03-01 00:03:09,3,1.63,1,N,186,107,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75
242510,2,2026-02-28 23:34:43,2026-02-28 23:50:47,3,1.02,1,N,79,249,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75
242511,2,2026-02-28 23:56:28,2026-03-01 00:08:07,4,1.66,1,N,79,90,1,12.1,1.0,0.5,0.00,0.0,1.0,17.85,2.5,0.0,0.75
